In [1]:
import pandas as pd
import numpy as np

# All ICU unit types

In [2]:
icustays = pd.read_csv("/home/ssimha/data/sagar-storage/mimiciv/3.1/icu/icustays.csv.gz")
transfers = pd.read_csv("/home/ssimha/data/sagar-storage/mimiciv/3.1/hosp/transfers.csv.gz")

In [3]:
icu_units = (
    icustays["first_careunit"]
    .dropna()
    .sort_values()
    .unique()
)

print(icu_units)

['Cardiac Vascular Intensive Care Unit (CVICU)' 'Coronary Care Unit (CCU)'
 'Intensive Care Unit (ICU)' 'Med/Surg'
 'Medical Intensive Care Unit (MICU)'
 'Medical/Surgical Intensive Care Unit (MICU/SICU)' 'Medicine'
 'Medicine/Cardiology Intermediate' 'Neuro Intermediate' 'Neuro Stepdown'
 'Neuro Surgical Intensive Care Unit (Neuro SICU)' 'Neurology' 'PACU'
 'Surgery/Trauma' 'Surgery/Vascular/Intermediate'
 'Surgical Intensive Care Unit (SICU)' 'Trauma SICU (TSICU)']


In [4]:
careunits = (
    transfers["careunit"]
    .dropna()
    .sort_values()
    .unique()
)

print(careunits)

['Cardiac Surgery' 'Cardiac Vascular Intensive Care Unit (CVICU)'
 'Cardiology' 'Cardiology Surgery Intermediate' 'Coronary Care Unit (CCU)'
 'Discharge Lounge' 'Emergency Department'
 'Emergency Department Observation' 'Hematology/Oncology'
 'Hematology/Oncology Intermediate' 'Intensive Care Unit (ICU)'
 'Labor & Delivery' 'Med/Surg' 'Med/Surg/GYN' 'Med/Surg/Trauma'
 'Medical Intensive Care Unit (MICU)' 'Medical/Surgical (Gynecology)'
 'Medical/Surgical Intensive Care Unit (MICU/SICU)' 'Medicine'
 'Medicine/Cardiology' 'Medicine/Cardiology Intermediate'
 'Neuro Intermediate' 'Neuro Stepdown'
 'Neuro Surgical Intensive Care Unit (Neuro SICU)' 'Neurology' 'Nursery'
 'Observation' 'Obstetrics (Postpartum & Antepartum)'
 'Obstetrics Antepartum' 'Obstetrics Postpartum' 'Oncology' 'PACU'
 'Psychiatry' 'Special Care Nursery (SCN)' 'Surgery'
 'Surgery/Pancreatic/Biliary/Bariatric' 'Surgery/Trauma'
 'Surgery/Vascular/Intermediate' 'Surgical Intensive Care Unit (SICU)'
 'Surgical Intermediate' 

In [5]:
transfers = transfers[
    ["subject_id", "hadm_id", "intime", "outtime", "careunit"]
].rename(
    columns={
        "intime": "transfer_intime",
        "outtime": "transfer_outtime"
    }
)

icustays["outtime"] = pd.to_datetime(icustays["outtime"])
transfers["transfer_intime"] = pd.to_datetime(transfers["transfer_intime"])
transfers["transfer_outtime"] = pd.to_datetime(transfers["transfer_outtime"])

# Merge ICU stays with transfers
df = icustays.merge(
    transfers,
    on=["subject_id", "hadm_id"],
    how="left"
)

# Keep only transfers at/after ICU outtime
df = df[df["transfer_intime"] >= df["outtime"]]

# Sort to identify earliest transfer after ICU
df = df.sort_values(["stay_id", "transfer_intime"])

# First transfer after ICU for each stay
first_transfer = df.groupby("stay_id", as_index=False).first()


# ICU vs non-ICU
ICU_UNITS = icu_units

def is_icu(careunit):
    if pd.isna(careunit):
        return False
    cu = str(careunit).upper()
    return any(unit in cu for unit in ICU_UNITS)

result = icustays.copy()

result = result.merge(
    first_transfer[["stay_id", "careunit"]],
    on="stay_id",
    how="left"
)

# ICU -> ICU transition
result["ICU_transfer"] = result["careunit"].apply(is_icu)

# ICU -> non-ICU ward
result["icu_to_ward"] = result["careunit"].notna() & (~result["ICU_transfer"])

# No transfer after ICU outtime
result["icu_mortality"] = result["careunit"].isna()


final_df = result[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "outtime",
        "careunit",
        "icu_to_ward",
        "ICU_transfer",
        "icu_mortality"
    ]
]

print(final_df[["icu_to_ward", "ICU_transfer", "icu_mortality"]].mean())
#print(final_df.head())

icu_to_ward      0.982786
ICU_transfer     0.017066
icu_mortality    0.000148
dtype: float64
